In [1]:
!pip uninstall -y pillow
!pip -q install pillow==9.5.0 pdfplumber sentence-transformers faiss-cpu transformers accelerate torch

Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 MB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 235.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 66.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires pillow>=10

In [3]:
import sys
!pip install --upgrade pillow>=10.1 --quiet
# Restart kernel to ensure changes take effect if necessary

import re
import faiss
import pdfplumber
import numpy as np

from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [5]:
print("Selecione um documento PDF.")

uploaded = files.upload()

NOME_PDF = list(uploaded.keys())[0]

print(f"Documento carregado: {NOME_PDF}")

Selecione um documento PDF.


Saving Política de Atendimento, trocas, devoluções e privacidade.pdf to Política de Atendimento, trocas, devoluções e privacidade.pdf
Documento carregado: Política de Atendimento, trocas, devoluções e privacidade.pdf


In [6]:
def limpar_texto(texto):

    texto = re.sub(r"\n+", "\n", texto)
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\s{2,}", " ", texto)

    return texto.strip()


def extrair_paginas(pdf_path):

    paginas = []

    with pdfplumber.open(pdf_path) as pdf:

        for numero, pagina in enumerate(pdf.pages, start=1):

            texto = pagina.extract_text()

            if texto:

                paginas.append({
                    "pagina": numero,
                    "texto": limpar_texto(texto)
                })

    return paginas


paginas = extrair_paginas(NOME_PDF)

print(f"{len(paginas)} páginas lidas.")

16 páginas lidas.


In [7]:
def criar_chunks_paginas(paginas,
                         tamanho=900,
                         sobreposicao_palavras=60):

    chunks = []

    for item in paginas:

        palavras = item["texto"].split()

        atual = []
        contador = 0

        for palavra in palavras:

            atual.append(palavra)
            contador += len(palavra) + 1

            if contador >= tamanho:

                chunks.append({
                    "pagina": item["pagina"],
                    "texto": " ".join(atual)
                })

                atual = atual[-sobreposicao_palavras:]
                contador = sum(len(p)+1 for p in atual)

        if atual:

            chunks.append({
                "pagina": item["pagina"],
                "texto": " ".join(atual)
            })

    return chunks


chunks = criar_chunks_paginas(paginas)

print(f"{len(chunks)} trechos criados.")

58 trechos criados.


In [8]:
print("Carregando modelo de embeddings...")

modelo_embedding = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

textos_chunks = [c["texto"] for c in chunks]

embeddings = modelo_embedding.encode(
    textos_chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

Carregando modelo de embeddings...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

print("Índice vetorial criado.")

Índice vetorial criado.


In [10]:
print("Carregando modelo de IA...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    truncation=True
)

Carregando modelo de IA...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxFo

In [11]:
def buscar_contexto(pergunta,
                    k=5,
                    limiar=0.32):

    pergunta_emb = modelo_embedding.encode(
        [pergunta],
        convert_to_numpy=True
    )

    faiss.normalize_L2(pergunta_emb)

    scores, ids = index.search(pergunta_emb, k)

    resultados = []

    for score, idx in zip(scores[0], ids[0]):

        if score >= limiar:

            resultados.append({
                "pagina": chunks[idx]["pagina"],
                "texto": chunks[idx]["texto"],
                "score": float(score)
            })

    return resultados

In [12]:
def responder(pergunta):

    resultados = buscar_contexto(pergunta)

    if len(resultados) == 0:
        return {
            "resposta": "Não encontrei essa informação no documento.",
            "paginas": []
        }

    contexto = ""

    paginas_utilizadas = []

    for r in resultados:

        paginas_utilizadas.append(r["pagina"])

        contexto += f"\n[PÁGINA {r['pagina']}]\n{r['texto']}\n"

    contexto = contexto[:2200]

    prompt = f"""

INSTRUÇÕES:

- Responda exclusivamente com base no contexto.
- Não invente informações.
- Seja objetivo.
- Quando possível, fundamente sua resposta no próprio texto.

CONTEXTO:

{contexto}

PERGUNTA:

{pergunta}

RESPOSTA:
"""

    resposta = llm(prompt)[0]["generated_text"].strip()

    return {
        "resposta": resposta,
        "paginas": sorted(set(paginas_utilizadas))
    }

In [13]:
print("="*60)
print("AGENTE DE IA PARA DOCUMENTOS PDF")
print("Digite 'sair' para encerrar.")
print("="*60)

while True:

    pergunta = input("\nPergunta: ")

    if pergunta.lower() == "sair":
        print("Sessão encerrada.")
        break

    resultado = responder(pergunta)

    print("\nResposta:\n")
    print(resultado["resposta"])

    if resultado["paginas"]:
        paginas = ", ".join(map(str, resultado["paginas"]))
        print(f"\nPáginas consultadas: {paginas}")

AGENTE DE IA PARA DOCUMENTOS PDF
Digite 'sair' para encerrar.

Pergunta: horário de atendimento


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Resposta:

INSTRUÇÕES:

- Responda exclusivamente com base no contexto.
- Não invente informações.
- Seja objetivo.
- Quando possível, fundamente sua resposta no próprio texto.

CONTEXTO:


[PÁGINA 3]
/ (22) 9XXXX-XXXX #### 2.1. SAC Presencial — Detalhamento Operacional O balcão de SAC (Serviço de Atendimento ao Consumidor) presencial funciona ininterruptamente, com escala de colaboradores em turnos de 6 horas, garantindo que nunca haja o posto desguarnecido, inclusive em feriados nacionais e municipais. SLA de Atendimento Presencial: • Tempo máximo de espera na fila: 5 minutos em horário de pico (18h–21h) e 2 minutos em horários normais. • Tempo médio de resolução de trocas simples (com nota fiscal e produto na embalagem): até 8 minutos. • Tempo médio de resolução de casos complexos (sem nota fiscal, produto avariado, necessidade de aprovação gerencial): até 20 minutos. Script Padrão de Abertura (SAC Presencial): > "Bom dia/Boa tarde/Boa noite! Bem-vindo(a) ao Mercado Central 24h. Me